pdf is a claude provided skill runs on server side

In [3]:
from pathlib import Path

from util.claude_client import  get_claude_client_and_default_model

client , llm = get_claude_client_and_default_model('claude')


llm profile name : claude , model : claude-haiku-4-5-20251001


In [4]:
message = client.messages.create(
    model = llm ,
    max_tokens = 16000,
    container={
        "skills": [{"type": "anthropic", "skill_id": "pdf", "version": "latest"}]
    },
    messages=[
        {
            "role": "user",
            "content": """Create a simple receipt PDF:

                RECEIPT

                Acme Corporation
                Date: January 15, 2025
                Receipt #: RCT-2025-001

                Customer: Jane Smith

                Items:
                - Product A: $50.00
                - Product B: $75.00
                - Product C: $25.00

                Subtotal: $150.00
                Tax (8%): $12.00
                Total: $162.00

                Thank you for your business!

                Use simple, clean formatting with clear sections.
                """,
        }
    ],
    tools=[{"type": "code_execution_20260521", "name": "code_execution"}]
)

file_id = None
for block in message.content:
    if block.type == "bash_code_execution_tool_result":
        if block.content.type == "bash_code_execution_result":
            for output in block.content.content:
                file_id = output.file_id


if file_id:
    # Download the file and save it
    output_path = Path.cwd() / "receipt.pdf"
    file_content = client.files.download(file_id=file_id)
    file_content.write_to_file(output_path)
    print(f"receipt saved to {output_path}")
else:
    raise RuntimeError(
        f"Claude returned no downloadable file (stop_reason={message.stop_reason}). "
        "Inspect message.content for tool errors or an unfinished generation."
    )

receipt saved to F:\projects\claude-client-sdk-example\src\skills\receipt.pdf
